In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.4_OHFP_alignment_reverse_onlyMat.py
-----------------------------------------------
4.2.4.4 の逆方向解析：フラグメント（FP） → 材料（OH）
- 各フラグメントを含むサンプル集合から、材料（OH変数）の出現分布を求め、
  それを OH クラスタ空間に写像して OH_purity / OH_entropy を算出。
- 同一 FP クラスタ内のフラグメントペアの「材料側での一貫性（cohesion）」を評価。
- FP クラスタ単位での「OH側一貫性（cohesive ratio）」を算出。
- 図表は章番号 4.2.4.4 を含む命名・フォルダで出力。本文用（cumeig×gap/db）に [Main] を付与。

前提（既存ファイル）:
  ROOT/
    ├─ features_OH_onlyMat.csv        # 行=sample_id, 列=材料の0/1
    ├─ features_FP_onlyMat.csv        # 行=sample_id, 列=フラグメントの0/1
    ├─ figs_OH/ClusterAssign_(top3|cumeig)_(index)_OH_*.csv
    └─ figs_FP/ClusterAssign_(top3|cumeig)_(index)_FP_*.csv

出力:
  ROOT/paper_4.2.4.4_OHFP/reverse/
    ├─ main_text/FigSet_4.2.4.4R_{mode}_{index}/…     # 本文推奨 (cumeig×gap, db)
    ├─ appendix/FigSet_4.2.4.4R_{mode}_{index}/…      # それ以外
    ├─ analysis_csv/
    │    ├─ Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv
    │    ├─ Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv
    │    └─ Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv
    └─ Table_4.2.4.4R_summary_all_conditions.csv
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
from itertools import combinations
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import cosine

# ========= パス設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"
FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

OUT_ROOT     = ROOT / "paper_4.2.4.4_OHFP" / "reverse"
OUT_MAIN     = OUT_ROOT / "main_text"
OUT_APPENDIX = OUT_ROOT / "appendix"
OUT_VARMAP   = OUT_ROOT / "analysis_csv"
for d in [OUT_ROOT, OUT_MAIN, OUT_APPENDIX, OUT_VARMAP]:
    d.mkdir(parents=True, exist_ok=True)

# ========= 解析パラメタ =========
DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]

# しきい値（安定化のため調整可）
MAT_PREV_THRESHOLD  = 0.20  # P(material=1 | fragment=1) を分布に採用する下限
PURITY_FLAG_THR     = 0.60  # “まとまる”と見なす目安（ペア評価で使用）
COS_SIM_THR         = 0.60
JS_DIV_THR          = 0.50

DPI = 300

# ========= フォント =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False

# ========= IO =========
def read_csv_guess(path: Path) -> pd.DataFrame:
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"Failed to read: {path}")

def load_features(path: Path) -> pd.DataFrame:
    df = read_csv_guess(path)
    if df.columns[0].lower() != "sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (X != 0).astype(int)

FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)

def latest_by_mode_index(fig_dir: Path, ds: str):
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds: 
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series:
    df = read_csv_guess(path)
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

# ========= 指標関数 =========
def entropy_from_dist(dist: dict[int, float]) -> float:
    if not dist: return np.nan
    p = np.array(list(dist.values()), dtype=float)
    p = p / (p.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def js_divergence(p, q):
    p = np.asarray(p, dtype=float); p = p/p.sum() if p.sum()>0 else p
    q = np.asarray(q, dtype=float); q = q/q.sum() if q.sum()>0 else q
    m = 0.5*(p+q)
    def _kl(a,b):
        a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1)
        return float(np.sum(a*(np.log2(a)-np.log2(b))))
    return 0.5*_kl(p,m) + 0.5*_kl(q,m)

# ========= フラグメント → OH分布 =========
def fragment_to_oh_distribution(fragment, Xfp, Xoh, oh_var2clu, threshold=MAT_PREV_THRESHOLD):
    idx = Xfp.index[Xfp[fragment] == 1]
    if len(idx) == 0:
        return None, None, 0, {}
    prev = Xoh.loc[idx].mean(axis=0)  # P(material=1 | fragment=1)
    sel = prev[prev >= threshold]
    if sel.empty:
        return None, None, len(idx), {}
    df = pd.DataFrame({"mat": sel.index, "w": sel.values})
    df["oh_cluster"] = df["mat"].map(oh_var2clu).astype("Int64")
    df = df.dropna(subset=["oh_cluster"])
    if df.empty:
        return None, None, len(idx), {}
    grp = df.groupby("oh_cluster", as_index=False)["w"].sum()
    grp["p"] = grp["w"]/grp["w"].sum()
    dist = dict(zip(grp["oh_cluster"].astype(int), grp["p"]))
    maj = int(grp.sort_values("p", ascending=False).iloc[0]["oh_cluster"])
    purity = float(grp["p"].max())
    return maj, purity, len(idx), dist

# ========= 図 =========
def plot_fragment_scatter(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
    set_font()
    if df_frag.empty: return
    size = 30 + 3 * df_frag["n_samples_with_fragment"].fillna(0).astype(float)
    c    = df_frag["FP_cluster"].astype("category")
    fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
    sc = ax.scatter(df_frag["OH_entropy"], df_frag["OH_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
    ax.set_xlabel("OH entropy (larger = more materials-spread)")
    ax.set_ylabel("OH purity (larger = more materials-specific)")
    title = f"Fig 4.2.4.4R: Fragments — Specificity vs Spread ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    lab_df = df_frag.sort_values(["OH_purity","n_samples_with_fragment"], ascending=False).head(top_labels)
    for _, r in lab_df.iterrows():
        ax.text(r["OH_entropy"], r["OH_purity"]+0.015, str(r["fragment"]), ha="center", va="bottom", fontsize=8)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragments_specificity-vs-spread_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_pair_distributions(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_pair is None or df_pair.empty: return
    set_font()
    # major same
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    rate = df_pair["OH_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
    idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
    ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
    ax.set_ylim(0,1); ax.set_ylabel("Proportion")
    title = f"Fig 4.2.4.4R: Pair-level — OH major same? ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, val in enumerate(vals):
        ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)
    # cosine / JS
    for col, xlabel in [("OH_cosine_similarity","Cosine similarity (1=similar)"),
                        ("OH_JS_divergence","JS divergence (0=similar)")]:
        s = df_pair[col].dropna()
        if not len(s): continue
        fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
        ax.hist(s, bins=20, alpha=0.9)
        title = f"Fig 4.2.4.4R: Pair-level — {col} ({mode} × {index})"
        if main_text: title += " [Main]"
        ax.set_title(title)
        ax.set_xlabel(xlabel); ax.set_ylabel("Count")
        fig.tight_layout()
        fname = f"Fig_4.2.4.4R_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
        fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
        fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
        plt.close(fig)

def plot_fpcluster_consistency(df_fpc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_fpc is None or df_fpc.empty: return
    set_font()
    df = df_fpc.sort_values("cohesive_ratio", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    x = np.arange(len(df))
    ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels([f"FP {int(c)}" for c in df["FP_cluster"]], rotation=35, ha="right")
    ax.set_ylim(0,1.05)
    ax.set_ylabel("Cohesive ratio (pair-level in OH space)")
    title = f"Fig 4.2.4.4R: FP-cluster cohesion in OH space ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, v in zip(x, df["cohesive_ratio"].values):
        ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_FPcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_mini_sankey(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, topN=20, main_text:bool=False):
    if df_frag is None or df_frag.empty: return
    set_font()
    df = df_frag.sort_values("OH_purity", ascending=False).head(topN).copy()
    if df.empty: return
    widths = (df["OH_purity"] + 0.2) / (df["OH_purity"].max() + 0.2)
    y = 0.0
    fig, ax = plt.subplots(figsize=(10, 0.35*len(df)+2), dpi=DPI)
    for i, r in df.iterrows():
        w = widths.loc[i]
        ax.plot([0, 1], [y, y], lw=8*w, alpha=0.9, color="#E15759")
        ax.text(-0.02, y, str(r["fragment"]), ha="right", va="center", fontsize=8)
        oh_major = r.get("OH_major_cluster")
        right_lbl = f"OH {int(oh_major)}" if pd.notna(oh_major) else "OH –"
        ax.text(1.02, y, right_lbl, ha="left", va="center", fontsize=8, color="#555")
        y += 0.5
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.5, y-0.0); ax.axis("off")
    title = f"Fig 4.2.4.4R: Fragment → OH major cluster (top {topN}) ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title, loc="left")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragment-to-OHmajor_miniAlluvial_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= メイン処理 =========
def main():
    set_font()

    # features（共通サンプルにアライン）
    Xoh0 = load_features(FEATURES_OH)
    Xfp0 = load_features(FEATURES_FP)
    common_samples = Xoh0.index.intersection(Xfp0.index)
    if len(common_samples)==0:
        raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
    Xoh0 = Xoh0.loc[common_samples]
    Xfp0 = Xfp0.loc[common_samples]

    # 既存クラスタ（最新）を (mode,index) で対応付け
    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys:
        raise RuntimeError("No common (mode,index) between figs_OH and figs_FP.")

    # 本文推奨（cumeig×gap, cumeig×db）
    main_pairs = {("cumeig","gap"), ("cumeig","db")}

    # 全条件のサマリ
    summary_rows = []

    for (mode, index) in keys:
        print(f"\n=== {mode} × {index} (reverse) ===")
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])

        # ---- フラグメント → OH 主分布 ----
        fragments = [f for f in fp_assign.index if f in Xfp0.columns]
        rows_frag = []
        for frag in fragments:
            n_frag = int(Xfp0[frag].sum())
            if n_frag <= 0:
                continue
            maj, purity, n_s, dist = fragment_to_oh_distribution(frag, Xfp0, Xoh0, oh_assign, threshold=MAT_PREV_THRESHOLD)
            fp_c = fp_assign.get(frag, np.nan)
            H = entropy_from_dist(dist)
            k_eff = len(dist) if dist else 0
            rows_frag.append({
                "mode": mode, "index": index,
                "fragment": frag,
                "FP_cluster": int(fp_c) if pd.notna(fp_c) else np.nan,
                "n_samples_with_fragment": n_s,
                "OH_major_cluster": maj,
                "OH_purity": purity,
                "OH_entropy": H,
                "OH_k_effective": k_eff,
                "OH_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
            })
        df_frag = pd.DataFrame(rows_frag)
        out_csv1 = OUT_VARMAP / f"Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv"
        if len(df_frag):
            df_frag.to_csv(out_csv1, index=False); print(f"[save] {out_csv1}")
        else:
            print("[info] no fragments mapped")

        # ---- フラグメントペア（同一FPクラスタ内）で材料側の一貫性を評価 ----
        rows_pair = []
        if len(df_frag):
            # OHクラスタ全集合のための埋め込み
            def parse_dist(s):
                if isinstance(s, str) and s:
                    d={}
                    for t in s.split(";"):
                        k,v = t.split(":"); d[int(k)] = float(v)
                    return d
                return {}
            dists = {r.fragment: parse_dist(r.OH_dist) for r in df_frag.itertuples()}
            all_oh_clusters = sorted({k for d in dists.values() for k in d.keys()})
            def vec_of(dist):
                return np.array([dist.get(k, 0.0) for k in all_oh_clusters], dtype=float)

            for k, grp in df_frag.dropna(subset=["FP_cluster"]).groupby("FP_cluster"):
                frags = grp["fragment"].tolist()
                for a, b in combinations(frags, 2):
                    da = dists.get(a, {}); db = dists.get(b, {})
                    va, vb = vec_of(da), vec_of(db)
                    if va.sum()==0 and vb.sum()==0:
                        cos_sim = np.nan; jsd = np.nan
                    else:
                        cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
                        jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
                    maj_same = (df_frag.loc[df_frag["fragment"]==a, "OH_major_cluster"].values[0] ==
                                df_frag.loc[df_frag["fragment"]==b, "OH_major_cluster"].values[0])
                    purity_min = float(min(
                        df_frag.loc[df_frag["fragment"]==a, "OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==a, "OH_purity"]) else 0.0,
                        df_frag.loc[df_frag["fragment"]==b, "OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==b, "OH_purity"]) else 0.0
                    ))
                    rows_pair.append({
                        "mode": mode, "index": index,
                        "FP_cluster": int(k),
                        "fragment_A": a, "fragment_B": b,
                        "OH_major_same": bool(maj_same),
                        "OH_purity_min": purity_min,
                        "OH_cosine_similarity": cos_sim,
                        "OH_JS_divergence": jsd
                    })
        df_pair = pd.DataFrame(rows_pair)
        out_csv2 = OUT_VARMAP / f"Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv"
        if len(df_pair):
            df_pair.to_csv(out_csv2, index=False); print(f"[save] {out_csv2}")

            # FPクラスタ単位の一貫性（材料側）
            def label_cohesion(r, purity_thr=PURITY_FLAG_THR, cos_thr=COS_SIM_THR, js_thr=JS_DIV_THR):
                return bool(
                    (r["OH_major_same"] and r["OH_purity_min"] >= purity_thr) or
                    (r["OH_cosine_similarity"] >= cos_thr) or
                    (pd.notna(r["OH_JS_divergence"]) and r["OH_JS_divergence"] <= js_thr)
                )
            df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
            df_fpc = (df_pair.groupby(["mode","index","FP_cluster"], as_index=False)
                             .agg(n_pairs=("cohesive_flag","size"),
                                  n_cohesive=("cohesive_flag","sum")))
            df_fpc["cohesive_ratio"] = df_fpc["n_cohesive"] / df_fpc["n_pairs"].replace(0, np.nan)
        else:
            df_fpc = pd.DataFrame()

        out_csv3 = OUT_VARMAP / f"Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv"
        if len(df_fpc):
            df_fpc.to_csv(out_csv3, index=False); print(f"[save] {out_csv3}")

        # ---- 図（本文/補遺の振り分けと命名）----
        is_main = (mode, index) in {("cumeig","gap"), ("cumeig","db")}
        out_dir = OUT_MAIN / f"FigSet_4.2.4.4R_{mode}_{index}" if is_main else OUT_APPENDIX / f"FigSet_4.2.4.4R_{mode}_{index}"
        out_dir.mkdir(parents=True, exist_ok=True)

        plot_fragment_scatter(df_frag, out_dir, mode, index, top_labels=15, main_text=is_main)
        plot_pair_distributions(df_pair, out_dir, mode, index, main_text=is_main)
        plot_fpcluster_consistency(df_fpc, out_dir, mode, index, main_text=is_main)
        plot_mini_sankey(df_frag, out_dir, mode, index, topN=20, main_text=is_main)
        print(f"[figs] saved under: {out_dir}")

        # ---- summary 行 ----
        summary_rows.append({
            "mode": mode, "index": index,
            "n_fragments": len(df_frag),
            "mean_OH_purity": df_frag["OH_purity"].dropna().mean() if "OH_purity" in df_frag else np.nan,
            "mean_OH_entropy": df_frag["OH_entropy"].dropna().mean() if "OH_entropy" in df_frag else np.nan,
            "pair_OH_major_same_rate": df_pair["OH_major_same"].mean() if len(df_pair) else np.nan,
            "pair_mean_cosine_similarity": df_pair["OH_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
            "pair_mean_JS_divergence": df_pair["OH_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
            "FPcluster_median_cohesive_ratio": df_fpc["cohesive_ratio"].median() if len(df_fpc) else np.nan
        })

    # ---- 全条件サマリー ----
    if summary_rows:
        df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
        outsum = OUT_ROOT / "Table_4.2.4.4R_summary_all_conditions.csv"
        df_sum.to_csv(outsum, index=False)
        print(f"[save] {outsum}")

    print("\n✅ Done: reverse-direction outputs →", OUT_ROOT)

if __name__ == "__main__":
    main()


=== top3 × silhouette (reverse) ===
[save] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-to-OHmajor_top3_silhouette.csv
[save] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-pair_cohesion_top3_silhouette.csv
[save] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_FPcluster_cohesion_top3_silhouette.csv
[figs] saved under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/appendix/FigSet_4.2.4.4R_top3_silhouette

=== top3 × dunn (reverse) ===
[save] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-to-OHmajor_top3_dunn.csv
[save] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv/Table_4.2.4.4R_fragment-pair_cohe